# 02 — Basic Rejection ABC

Infer (beta, gamma, rho) using rejection ABC with MAD-normalised Euclidean distance.

**Experiments:**
1. Run 50,000 simulations with all 12 summary statistics
2. Acceptance quantile sweep (0.5%, 1%, 2%, 5%)
3. Corner plots of approximate posterior
4. Posterior predictive checks

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import corner
import time

from data_loader import load_all
from summary_stats import compute_observed_summaries, compute_summaries, SUMMARY_NAMES, IDX_ALL
from abc_rejection import run_rejection_abc, accept_quantile
from simulator import simulate_fast

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

In [ ]:
# Load observed data and compute summary statistics
inf_ts, rew_ts, deg_hist = load_all()
s_obs, s_per_rep = compute_observed_summaries(inf_ts, rew_ts, deg_hist)
print('Observed summaries:', np.round(s_obs, 4))

## 2.1 Run Rejection ABC (50,000 simulations)

In [ ]:
N_SIM = 50_000

# Warm up Numba JIT
_ = simulate_fast(0.2, 0.1, 0.3, seed=0)

t0 = time.time()
thetas, distances, summaries, mads = run_rejection_abc(
    s_obs=s_obs,
    n_sim=N_SIM,
    simulator_fn=simulate_fast,
    indices=IDX_ALL,
    seed=42,
    verbose=True,
)
elapsed = time.time() - t0
print(f'\nCompleted {N_SIM} simulations in {elapsed:.1f}s ({elapsed/N_SIM*1000:.2f}ms each)')

# Save results
os.makedirs('../results', exist_ok=True)
np.savez('../results/rejection_abc_50k.npz',
         thetas=thetas, distances=distances, summaries=summaries, mads=mads, s_obs=s_obs)

## 2.2 Acceptance Quantile Sweep

In [ ]:
quantiles = [0.005, 0.01, 0.02, 0.05]
param_names = [r'$\beta$', r'$\gamma$', r'$\rho$']
prior_bounds = [(0.05, 0.50), (0.02, 0.20), (0.0, 0.80)]

fig, axes = plt.subplots(len(quantiles), 3, figsize=(14, 3.5 * len(quantiles)))

for row, q in enumerate(quantiles):
    acc_theta, acc_d, acc_s, threshold = accept_quantile(thetas, distances, summaries, q)
    n_acc = len(acc_theta)

    for col in range(3):
        ax = axes[row, col]
        ax.hist(acc_theta[:, col], bins=30, density=True, color='steelblue', alpha=0.7, edgecolor='white')
        lo, hi = prior_bounds[col]
        ax.axhline(1.0 / (hi - lo), color='grey', linestyle='--', alpha=0.5, label='Prior')
        ax.set_xlabel(param_names[col])
        if col == 0:
            ax.set_ylabel(f'q={q} (n={n_acc})')
        if row == 0:
            ax.set_title(param_names[col])
        ax.legend(fontsize=8)

plt.suptitle('Rejection ABC: Posterior Marginals at Different Acceptance Quantiles', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/rejection_abc_quantile_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.3 Corner Plot (q=1%)

In [ ]:
acc_theta, acc_d, acc_s, threshold = accept_quantile(thetas, distances, summaries, quantile=0.01)

print(f'Accepted {len(acc_theta)} samples (threshold = {threshold:.3f})')
print(f'Posterior means:   beta={acc_theta[:,0].mean():.4f}, gamma={acc_theta[:,1].mean():.4f}, rho={acc_theta[:,2].mean():.4f}')
print(f'Posterior medians: beta={np.median(acc_theta[:,0]):.4f}, gamma={np.median(acc_theta[:,1]):.4f}, rho={np.median(acc_theta[:,2]):.4f}')

for i, name in enumerate(param_names):
    q025, q975 = np.percentile(acc_theta[:, i], [2.5, 97.5])
    print(f'  {name}: 95% CI = [{q025:.4f}, {q975:.4f}]')

fig = corner.corner(
    acc_theta,
    labels=[r'$\beta$', r'$\gamma$', r'$\rho$'],
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True,
    title_kwargs={'fontsize': 12},
    color='steelblue',
)
plt.suptitle('Rejection ABC Posterior (q=1%, all 12 summaries)', fontsize=14, y=1.02)
plt.savefig('../figures/rejection_abc_corner.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.4 Posterior Summary Table

In [ ]:
header = f"{'Quantile':>10} {'N_acc':>8} {'beta_mean':>10} {'gamma_mean':>11} {'rho_mean':>10} {'beta_95CI':>16} {'gamma_95CI':>16} {'rho_95CI':>16}"
print(header)
print('-' * len(header))
for q in quantiles:
    at, ad, asum, thr = accept_quantile(thetas, distances, summaries, q)
    row = f'{q:>10.3f} {len(at):>8d}'
    for i in range(3):
        row += f' {at[:,i].mean():>10.4f}'
    for i in range(3):
        lo, hi = np.percentile(at[:, i], [2.5, 97.5])
        row += f' [{lo:.4f},{hi:.4f}]'
    print(row)

## 2.5 Posterior Predictive Check

Simulate from accepted parameters and overlay on observed data to verify the posterior produces data consistent with observations.

In [ ]:
# Posterior predictive check: simulate from 20 accepted parameter draws
acc_theta_ppc, _, _, _ = accept_quantile(thetas, distances, summaries, quantile=0.01)
n_ppc = min(20, len(acc_theta_ppc))
rng_ppc = np.random.default_rng(99)
idx_ppc = rng_ppc.choice(len(acc_theta_ppc), size=n_ppc, replace=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
t_arr = np.arange(inf_ts.shape[1])

# Plot observed
for r in range(inf_ts.shape[0]):
    axes[0].plot(t_arr, inf_ts[r], color='steelblue', alpha=0.15)
    axes[1].plot(t_arr, rew_ts[r], color='seagreen', alpha=0.15)

axes[2].bar(np.arange(31), np.mean(deg_hist, axis=0), alpha=0.3, color='steelblue', label='Observed mean')

# Plot posterior predictive simulations
for i in idx_ppc:
    b, g, r = acc_theta_ppc[i]
    inf_sim, rew_sim, deg_sim = simulate_fast(b, g, r, seed=int(rng_ppc.integers(0, 2**31)))
    axes[0].plot(t_arr, inf_sim, color='red', alpha=0.3, linewidth=0.8)
    axes[1].plot(t_arr, rew_sim, color='red', alpha=0.3, linewidth=0.8)

# Degree: overlay one posterior predictive draw
b, g, r = acc_theta_ppc[idx_ppc[0]]
_, _, deg_ppc = simulate_fast(b, g, r, seed=12345)
axes[2].bar(np.arange(31), deg_ppc, alpha=0.5, color='red', label='Posterior pred.')

axes[0].set_title('Infected fraction')
axes[0].set_xlabel('Time')
axes[1].set_title('Rewiring counts')
axes[1].set_xlabel('Time')
axes[2].set_title('Final degree histogram')
axes[2].set_xlabel('Degree')
axes[2].legend()

# Custom legend
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], color='steelblue', alpha=0.5, label='Observed'),
                   Line2D([0], [0], color='red', alpha=0.5, label='Posterior predictive')]
axes[0].legend(handles=legend_elements, fontsize=9)

plt.suptitle('Posterior Predictive Check (Rejection ABC, q=1%)', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/rejection_abc_ppc.png', dpi=150, bbox_inches='tight')
plt.show()